# Titanic: множества W и P

**Цель:** для двух заданных вариантов выделить результативные признаки W, сформировать P без дублей и оценить силу и направление наблюдаемых связей. Источник — `seaborn.load_dataset("titanic")`; при отсутствии доступа к сети используется официальный CSV в папке проекта.

Снижение ошибки оценивается на пяти проверочных блоках для каждого P отдельно. Это мера прогностической связи, а не причинного действия.


In [ ]:
from pathlib import Path
import sys

import pandas as pd

project = Path.cwd()
if (project / "titanic_w_p").is_dir():
    sys.path.insert(0, str(project / "titanic_w_p"))
else:
    sys.path.insert(0, str(project))

from analysis import DUPLICATES, TARGETS, group_summary, load_data, make_sets, run

df, source = load_data()
print("Источник:", source)
print("Размер:", df.shape)
display(df.head())


## 1. Проверка данных и формирование W и P

`alive`, `class`, `embark_town`, `who`, `adult_male`, `alone` исключены из P как дубли или производные признаки. Целевые столбцы также не попадают в P своего варианта.


In [ ]:
print("Пропуски по столбцам:")
display(df.isna().sum().to_frame("missing"))

print("Исключённые дубли:")
for duplicate, original in DUPLICATES.items():
    print(f"{duplicate} → {original}")

for name, targets in TARGETS.items():
    W, P = make_sets(df, targets)
    print(f"\n{name}: W = {W}; P = {P}")


## 2. Ранжирование показателей

Оценка — сокращение средней квадратичной ошибки относительно прогноза средним обучающей выборки. Для бинарных `survived` и `sex` это Brier score. Значения ниже нуля означают отсутствие пользы по сравнению с базовым прогнозом.


In [ ]:
results = run(df)

for name, variant in results["variants"].items():
    print(f"\n{name}")
    ranking = pd.DataFrame(variant["ranking"])
    for target in variant["W"]:
        print(f"Цель: {target}; наблюдений: {df[target].notna().sum()}")
        table = ranking[ranking["target"] == target].sort_values("gain", ascending=False).copy()
        table["снижение ошибки, %"] = (100 * table.pop("gain")).round(1)
        display(table.rename(columns={"feature": "показатель P", "observed_n": "без пропусков в паре"}).drop(columns="target"))


## 3. Направление наблюдаемой связи

Для `survived` выводится доля выживших; для `sex` — доля женщин; для `age` — средний возраст. Группы с малой численностью и `deck` с 688 пропусками требуют осторожной трактовки.


In [ ]:
examples = {
    "survived": ["sex", "pclass", "fare", "age", "sibsp", "parch", "embarked", "deck"],
    "age": ["pclass", "parch", "sibsp", "sex"],
    "sex": ["fare", "parch", "sibsp", "pclass"],
}
for target, features in examples.items():
    print(f"\nЦель: {target}")
    for feature in features:
        print(f"\n{feature}")
        display(group_summary(df, target, feature).round(3))


## Выводы

1. **Вариант 1:** для выживания сильнее всего `sex` (29,2 % сокращения ошибки), для возраста — `pclass` (13,5 %).
2. **Вариант 2:** после переноса `sex` в W главным одиночным показателем выживания становится `pclass` (10,7 %); для пола — `fare` (6,9 %).
3. Женщины выживали чаще мужчин (74,2 % против 18,9 %). Выживаемость снижалась от 1 к 3 классу (63,0 % → 24,2 %). Возраст в среднем выше у пассажиров 1 класса (38,2 года), чем 3 класса (25,1 года).
4. Эти связи не доказывают причинное влияние. `pclass`, `fare` и наличие сведений о палубе взаимосвязаны; результаты по возрасту основаны на 714 наблюдениях, по известной палубе — на 203.
